[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/13_gpt2_block_solution.ipynb)

# ✅ Solution: gpt2_block

Implement a full **GPT-2 style Transformer block** — combining everything you've learned.

### Architecture (Pre-Norm)
```
x = x + causal_self_attention(ln1(x))
x = x + mlp(ln2(x))
```

### Signature
```python
class GPT2Block(nnx.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: jax.Array) -> jax.Array: ...
```

### Requirements
- Inherit from `nnx.Module`
- `self.ln1`, `self.ln2`: `nn.LayerNorm(d_model)`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` for attention
- `self.mlp`: `nn.Sequential(Linear(d, 4d), GELU(), Linear(4d, d))`
- Attention must be **causal** (mask future positions)
- Pre-norm architecture (LayerNorm *before* attention and MLP)
- Residual connections around both attention and MLP


In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge')
except ImportError:
    pass


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import math


In [ ]:
# ✅ SOLUTION

import jax, jax.numpy as jnp, math
from flax import nnx
class _MLP(nnx.Module):
    def __init__(self, d, *, rngs):
        self.fc = nnx.Linear(d, 4 * d, rngs=rngs)
        self.proj = nnx.Linear(4 * d, d, rngs=rngs)
    def __call__(self, x_BLD):
        return self.proj(jax.nn.gelu(self.fc(x_BLD)))
class GPT2Block(nnx.Module):
    """B=batch, L=seq, D=d_model, H=heads, K=d_k."""
    def __init__(self, d_model, num_heads, *, rngs):
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.ln1 = nnx.LayerNorm(d_model, rngs=rngs)
        self.ln2 = nnx.LayerNorm(d_model, rngs=rngs)
        self.W_q = nnx.Linear(d_model, d_model, rngs=rngs)
        self.W_k = nnx.Linear(d_model, d_model, rngs=rngs)
        self.W_v = nnx.Linear(d_model, d_model, rngs=rngs)
        self.W_o = nnx.Linear(d_model, d_model, rngs=rngs)
        self.mlp = _MLP(d_model, rngs=rngs)
    def _attn(self, x_BLD):
        B, L, _ = x_BLD.shape
        H, K = self.num_heads, self.d_k
        q_BHLK = self.W_q(x_BLD).reshape(B, L, H, K).transpose(0, 2, 1, 3)
        k_BHLK = self.W_k(x_BLD).reshape(B, L, H, K).transpose(0, 2, 1, 3)
        v_BHLK = self.W_v(x_BLD).reshape(B, L, H, K).transpose(0, 2, 1, 3)
        scores_BHLL = q_BHLK @ jnp.swapaxes(k_BHLK, -2, -1) / math.sqrt(K)
        causal_LL = jnp.triu(jnp.ones((L, L), dtype=bool), 1)
        attn_BHLK = jax.nn.softmax(jnp.where(causal_LL, -jnp.inf, scores_BHLL), axis=-1) @ v_BHLK
        concat_BLD = attn_BHLK.transpose(0, 2, 1, 3).reshape(B, L, H * K)
        return self.W_o(concat_BLD)
    def __call__(self, x_BLD):
        x_BLD = x_BLD + self._attn(self.ln1(x_BLD))
        return x_BLD + self.mlp(self.ln2(x_BLD))


In [ ]:
# Verify
print(GPT2Block)


In [ ]:
from jax_judge import check
check("gpt2_block")
